In [1]:
# Run on GPU server (Jupyter). Local CPU will OOM at L=2520.
import os
import sys
from pathlib import Path

import torch

# Locate repo root
_env = os.environ.get("SYNTHGEN_ROOT", "")
REPO = Path(_env) if _env else None
if REPO is None or not (REPO / "SBBTS").is_dir():
    for _c in [
        Path.home() / "SyntheticGenerators",
        Path.cwd(),
        Path.cwd().parent,
    ]:
        if (_c / "SBBTS").is_dir():
            REPO = _c
            break
    else:
        raise RuntimeError(
            "Repo not found. Set SYNTHGEN_ROOT or clone to ~/SyntheticGenerators."
        )

SBBTS = REPO / "SBBTS"
for p in (SBBTS, REPO):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

if not torch.cuda.is_available():
    raise RuntimeError("CUDA required for SBBTS training (seq_len=2520).")

device = torch.device("cuda")
OUTPUT_DIR     = SBBTS / "output_data"          # train_normalized.npy lives here
TRAIN_NPY      = OUTPUT_DIR / "train_normalized.npy"
CHECKPOINT_DIR = SBBTS / "checkpoints" / "sbbts_run"

print(f"repo        = {REPO}")
print(f"SBBTS       = {SBBTS}")
print(f"OUTPUT_DIR  = {OUTPUT_DIR}")
print(f"TRAIN_NPY   = {TRAIN_NPY}  exists={TRAIN_NPY.exists()}")
print(f"device      = {device} ({torch.cuda.get_device_name(0)})")

repo        = /home/jovyan/SyntheticGenerators
SBBTS       = /home/jovyan/SyntheticGenerators/SBBTS
OUTPUT_DIR  = /home/jovyan/SyntheticGenerators/SBBTS/output_data
TRAIN_NPY   = /home/jovyan/SyntheticGenerators/SBBTS/output_data/train_normalized.npy  exists=True
device      = cuda (NVIDIA A16)


In [ ]:
from adapter_sbbts import build_sbbts_tensor, train_sbbts

# 14-16GB GPU: batch_size=1-2. 24GB+: try 4.
BATCH_SIZE = 2
N_EPOCHS   = 1000
K          = 5

X, scale, idx, meta = build_sbbts_tensor(
    train_npy_path=TRAIN_NPY,
    M_train=500,
    device=device,
)

model, y_0 = train_sbbts(
    X=X,
    scale=scale,
    checkpoint_dir=CHECKPOINT_DIR,
    device=device,
    batch_size=BATCH_SIZE,
    n_epochs=N_EPOCHS,
    k=K,
)
# checkpoint saved to CHECKPOINT_DIR/best_sbbts.pt
# per-iteration snapshots at CHECKPOINT_DIR/iter_1.pt, iter_2.pt, ...
print("Training complete. Checkpoint:", CHECKPOINT_DIR / "best_sbbts.pt")

In [ ]:
from adapter_sbbts import sample_synthetic

synthetic = sample_synthetic(
    X=X,
    model=model,
    y_0=y_0,
    scale=scale,
    meta=meta,
    output_dir=OUTPUT_DIR,
    M_simu=200,
    device=device,
)
print("benchmark export:", OUTPUT_DIR / "sbbts_synthetic.npy")

In [ ]:
# ---- Run this cell if training finished but sampling failed ----
# Reloads the final checkpoint and generates without re-training.
from adapter_sbbts import load_and_generate

synthetic = load_and_generate(
    checkpoint_dir=CHECKPOINT_DIR,
    train_npy_path=TRAIN_NPY,
    output_dir=OUTPUT_DIR,
    M_simu=200,
    device=device,
)
print("benchmark export:", OUTPUT_DIR / "sbbts_synthetic.npy")

In [ ]:
# Optional: smoke test (2 epochs, K=5) — checks detach fix before full run
from adapter_sbbts import build_sbbts_tensor, train_sbbts

if "X" not in dir():
    X, scale, idx, meta = build_sbbts_tensor(TRAIN_NPY, M_train=500, device=device)

model_smoke, y_0_smoke = train_sbbts(
    X=X, scale=scale,
    checkpoint_dir=CHECKPOINT_DIR / "smoke",
    device=device, batch_size=2, n_epochs=2, k=5,
)